# Web Scraping from Youtube

In [ ]:
from googleapiclient.discovery import build
import pandas as pd
import time

# Replace with your API key
API_KEY = "//your_youtube_api_key//"
youtube = build("youtube", "v3", developerKey=API_KEY)

# Function to get video IDs for political/news videos
def get_video_ids(query="politics news", max_pages=2, results_per_page=20):
    video_ids = []
    next_page_token = None
    
    for _ in range(max_pages):
        request = youtube.search().list(
            q=query,
            part="snippet",
            type="video",
            maxResults=results_per_page,
            pageToken=next_page_token
        )
        response = request.execute()
        
        for item in response["items"]:
            video_ids.append(item["id"]["videoId"])
        
        next_page_token = response.get("nextPageToken")
        if not next_page_token:
            break
        time.sleep(1)
    
    return video_ids

# Function to fetch comments from a video
def get_comments(video_id, max_comments=50):
    comments = []
    request = youtube.commentThreads().list(
        part="snippet",
        videoId=video_id,
        maxResults=100,
        textFormat="plainText"
    )
    response = request.execute()
    
    for item in response["items"]:
        comment = item["snippet"]["topLevelComment"]["snippet"]
        comments.append({
            "video_id": video_id,
            "author": comment["authorDisplayName"],
            "comment": comment["textDisplay"],
            "likes": comment["likeCount"],
            "published_at": comment["publishedAt"]
        })
        if len(comments) >= max_comments:
            break
    
    return comments

# 🔹 Main
video_ids = get_video_ids(query="politics news", max_pages=3, results_per_page=20)
print(f"Found {len(video_ids)} videos")

all_comments = []
for vid in video_ids:
    try:
        video_comments = get_comments(vid, max_comments=100)  # limit 100 comments/video
        all_comments.extend(video_comments)
        time.sleep(1)
    except:
        continue

# Save to CSV
df = pd.DataFrame(all_comments)
df.to_csv("political_news_comments.csv", index=False)

print("✅ Comments dataset saved: political_news_comments.csv")


Found 60 videos
✅ Comments dataset saved: political_news_comments.csv


In [4]:
import pandas as pd

# Load the scraped file
df = pd.read_csv("political_news_comments.csv")

# Keep only the 'comment' column
df = df[["comment"]]

# Optional: clean text (remove links, make lowercase)
df["comment"] = df["comment"].str.lower().str.replace(r"http\S+|www\S+", "", regex=True)

# Save as new dataset
df.to_csv("political_news_comments_only.csv", index=False)

print("✅ Clean dataset saved: political_news_comments_only.csv")


✅ Clean dataset saved: political_news_comments_only.csv


In [5]:
!pip install langdetect deep-translator


Defaulting to user installation because normal site-packages is not writeable



[notice] A new release of pip is available: 25.1 -> 25.2
[notice] To update, run: python.exe -m pip install --upgrade pip


In [6]:
from deep_translator import GoogleTranslator
from langdetect import detect
import pandas as pd

# Load scraped dataset
df = pd.read_csv("political_news_comments_only.csv")

def translate_to_english(text):
    try:
        if detect(text) != "en":  # if not English
            return GoogleTranslator(source='auto', target='en').translate(text)
        else:
            return text
    except:
        return text  # fallback if detection/translation fails

# Apply translation
df['comment_translated'] = df['comment'].astype(str).apply(translate_to_english)

# Save translated dataset
df.to_csv("political_comments_translated.csv", index=False)
print("✅ Translated dataset saved as political_news_comments_translated.csv")

✅ Translated dataset saved as political_news_comments_translated.csv


In [8]:
import pandas as pd
import re
import string
import nltk

# Download stopwords if not already
nltk.download('stopwords')
from nltk.corpus import stopwords

# Load translated dataset
df = pd.read_csv("political_comments_translated.csv")

# Define preprocessing function
def preprocess_text(text):
    if not isinstance(text, str):
        return ""
    
    # 1. Lowercase
    text = text.lower()
    
    # 2. Remove URLs
    text = re.sub(r"http\S+|www\S+|https\S+", "", text)
    
    # 3. Remove mentions (@username) and hashtags
    text = re.sub(r"@\w+|#\w+", "", text)
    
    # 4. Remove punctuation
    text = text.translate(str.maketrans("", "", string.punctuation))
    
    # 5. Remove numbers
    text = re.sub(r"\d+", "", text)
    
    # 6. Remove extra whitespaces
    text = " ".join(text.split())
    
    # 7. Remove stopwords (optional)
    stop_words = set(stopwords.words("english"))
    text = " ".join([word for word in text.split() if word not in stop_words])
    
    return text

# Apply preprocessing to translated comments
df["comment_cleaned"] = df["comment_translated"].apply(preprocess_text)

# Save preprocessed dataset
df.to_csv("political_news_comments_preprocessed.csv", index=False)
print("✅ Preprocessed dataset saved as political_news_comments_preprocessed.csv")

# Show sample
print(df[["comment_translated", "comment_cleaned"]].head(10))


[nltk_data] Downloading package stopwords to
[nltk_data]     C:\Users\Muktha\AppData\Roaming\nltk_data...
[nltk_data]   Package stopwords is already up-to-date!


✅ Preprocessed dataset saved as political_news_comments_preprocessed.csv
                                  comment_translated  \
0                           remove rss modi agenda 😂   
1  pm pays tribute to mahatma on the independence...   
2  speaking about rss on independence day. what a...   
3  no more a hidden agendas. modi's last fight wh...   
4  where is kangress ka pappu bhakths 😂 seems lik...   
5  who cares what anti national deep state backed...   
6  r s s is a great organization. 100 years of gr...   
7                                        Ha ha Ha !!   
8  no role of rss in freedom movement \nthey didn...   
9                  illiterate annoying ignorance 😂😂😂   

                                     comment_cleaned  
0                           remove rss modi agenda 😂  
1  pm pays tribute mahatma independence day prais...  
2            speaking rss independence day joke 😂😂😂😂  
3  hidden agendas modis last fight bjp n rss goes...  
4  kangress ka pappu bhakths 😂 seem

In [3]:
import pandas as pd
import re

# Load your preprocessed dataset
df = pd.read_csv("political_news_comments_preprocessed.csv")

# Function to remove emojis
def remove_emojis(text):
    emoji_pattern = re.compile(
        "["
        u"\U0001F600-\U0001F64F"  # emoticons
        u"\U0001F300-\U0001F5FF"  # symbols & pictographs
        u"\U0001F680-\U0001F6FF"  # transport & map symbols
        u"\U0001F1E0-\U0001F1FF"  # flags
        u"\U00002702-\U000027B0"
        u"\U000024C2-\U0001F251"
        "]+", flags=re.UNICODE
    )
    return emoji_pattern.sub(r'', str(text))  # make sure it's string

# Apply emoji removal
df['comment_cleaned'] = df['comment_cleaned'].apply(remove_emojis)

# Save cleaned dataset
df.to_csv("political_news_comments_preprocessed.csv", index=False)

print("✅ Emojis removed and file updated: political_news_comments_preprocessed.csv")


✅ Emojis removed and file updated: political_news_comments_preprocessed.csv
